# Проба S2DR3: Київ, 4 × 4 км

Цільова дата: **31.10.2026**. Модель самостійно вибирає сцену в доступному періоді; фактичну дату дивіться у результаті.

1. Виберіть **Runtime → Change runtime type → T4 GPU**.
2. Увійдіть у [Gamma Earth](https://app.gamma.earth/), скопіюйте свій активний API key.
3. У **Secrets** Google Colab створіть `GE_API_KEY` і дозвольте доступ цьому блокноту. Не вставляйте ключ у клітинки.
4. Виконуйте клітинки по черзі. Запускається одне завдання на ділянку 16 км² (орієнтовно 8 безкоштовних кредитів для України; перевірте баланс).

Зображення 1 м/піксель відновлює ШІ; можливі спотворення дрібних об'єктів. Проба призначена для оцінки якості. Безкоштовні прев'ю можуть бути публічними.

[Офіційна інструкція](https://help.gamma.earth/docs/google-colab/connect-your-account-to-google-colab/) · [Офіційний блокнот і ліцензія Gamma Earth](https://colab.research.google.com/drive/18phbwA1iYG5VDGN2WjK7WrWYi-FdCHJ5)


In [ ]:
import sys
if sys.version_info[:2] != (3, 13):
    raise RuntimeError("Ця версія S2DR3 потребує Python 3.13 у Google Colab.")
%pip install -q https://storage.googleapis.com/0x7ff601307fa5/s2dr3-20261012.1-cp313-cp313-linux_x86_64.whl


In [ ]:
import os
from google.colab import userdata

try:
    gamma_key = userdata.get("GE_API_KEY")
except Exception:
    raise RuntimeError("Додайте GE_API_KEY у Secrets Colab і дозвольте доступ цьому блокноту.") from None
if not gamma_key or not gamma_key.strip():
    raise RuntimeError("Секрет GE_API_KEY порожній.")
os.environ["GE_API_KEY"] = gamma_key.strip()
del gamma_key
print("Ключ завантажено із Secrets; його значення приховано.")


In [ ]:
import s2dr3.inferutils

# Координати у порядку довгота, широта. Стандартне демо обробляє 4 × 4 км.
lonlat = (30.523404, 50.450099)
requested_date = "2026-10-31"

print(f"Проба: центр Києва, 4 × 4 км; цільова дата {requested_date}.")
print("Фактична дата знімка може відрізнятися; перевірте її в результаті.")
try:
    s2dr3.inferutils.test(lonlat, requested_date)
except Exception as error:
    message = str(error).replace(os.environ["GE_API_KEY"], "[ключ приховано]")
    raise RuntimeError(message) from None


## Експорт для карти

Після успішної обробки відкрийте посилання прев'ю й оцініть результат. Наступна клітинка експортує лише поліпшені RGB GeoTIFF та опис запиту. Ключ у ZIP не потрапляє. GeoTIFF ще потрібно перевірити й перетворити на тайли перед додаванням у карту.


In [ ]:
import json
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile
from google.colab import files

output_dir = Path("/content/output")
products = sorted(output_dir.rglob("S2L2Ax10_*_TCI.tif"))
if not products:
    raise RuntimeError("Поліпшених RGB GeoTIFF ще немає. Спершу завершіть обробку.")
archive_path = Path("/content/s2dr3-kyiv-trial.zip")
request_info = {
    "model": "S2DR3",
    "sdk": "20261012.1",
    "center_lonlat": list(lonlat),
    "area_km2": 16,
    "requested_date": requested_date,
    "note": "Target date is not the acquisition date. Read the actual date from each generated product.",
    "products": [str(product.relative_to(output_dir)) for product in products],
}
with ZipFile(archive_path, "w", ZIP_DEFLATED) as archive:
    for product in products:
        archive.write(product, str(product.relative_to(output_dir)))
    archive.writestr("trial-request.json", json.dumps(request_info, ensure_ascii=False, indent=2))
print(f"Готово до експорту: {len(products)} RGB GeoTIFF.")
files.download(str(archive_path))
